# 🐍 Master Guide: `asyncio`, Coroutines & Cooperative Multitasking
> **Single-Source Revision Guide**: Covers the event loop, `async`/`await` syntax, `asyncio.gather()`, Task scheduling, and blocking pitfalls.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
`asyncio` enables high-concurrency single-threaded I/O using an **Event Loop**. Coroutines cooperate by yielding control (`await`) while waiting for network sockets or timer ticks, handling 100,000+ connections with minimal RAM overhead.

---
## 🧠 2. CPython Architecture: How Coroutines Work

### Coroutines are Enhanced Generators
Under the hood, native coroutines (`async def`) are implemented on top of generator infrastructure in CPython (`PyCoroObject`).

```
┌──────────────────────────────────────────────────────────┐
|                  The asyncio Event Loop                |
|                                                        |
|   Task A (awaits HTTP) ──> Pauses, yields to Event Loop|
|   Event Loop checks OS socket polling (epoll/kqueue/IOCP)|
|   Task B resumes ────────> Runs until next await       |
└──────────────────────────────────────────────────────────┘
```

- **Cooperative vs Preemptive**: OS threads are preemptively interrupted by the OS scheduler. Async coroutines are cooperative: a coroutine ONLY gives up control when it hits an explicit `await`!

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: `async def` and `await`
In modern Jupyter Notebooks, the event loop is already running! You can use `await` directly at the top level.

In [ ]:
import asyncio

async def fetch_data(source, delay):
    print(f"Fetching from {source}...")
    await asyncio.sleep(delay)  # Non-blocking pause
    print(f"Received data from {source}!")
    return f"Data({source})"

# Direct top-level await in Jupyter:
result = await fetch_data("Weather API", 0.1)
print("Result:", result)

---
## 🟡 4. Part 2: Intermediate Features

### Concurrent Execution with `asyncio.gather()`
Fires multiple coroutines concurrently and collects all results.

In [ ]:
import time

async def main_tasks():
    t0 = time.perf_counter()
    # Run 3 network calls concurrently:
    results = await asyncio.gather(
        fetch_data("Database", 0.2),
        fetch_data("Redis Cache", 0.1),
        fetch_data("Auth Service", 0.15)
    )
    t1 = time.perf_counter()
    print(f"\nAll 3 tasks finished in {(t1 - t0)*1000:.1f} ms (Ran concurrently!)")
    return results

all_data = await main_tasks()
print("Collected results:", all_data)

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: The Blocking Call Disaster (`time.sleep`)
Calling synchronous blocking functions (like `time.sleep()`, standard `requests.get()`, or heavy CPU loops) inside an async function **freezes the entire event loop**, freezing all other concurrent tasks!

In [ ]:
# To safely run blocking legacy functions, offload them to an executor thread:
def blocking_io():
    import time
    time.sleep(0.1)
    return "Finished blocking work"

# asyncio.to_thread runs blocking code in a worker thread pool (Python 3.9+):
safe_result = await asyncio.to_thread(blocking_io)
print("Safely executed blocking code via asyncio.to_thread:", safe_result)

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| async def                          | Declares a native coroutine           |
| await coro                         | Yields control to loop until complete |
| asyncio.gather(*coros)             | Runs coroutines concurrently          |
| asyncio.run(main())                | Entrypoint in standard scripts        |
| asyncio.to_thread(blocking_fn)     | Offloads blocking I/O to thread pool  |
| time.sleep() in async              | FATAL MISTAKE! Freezes the event loop |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. High-Throughput Concurrent Service Simulation (`asyncio.gather` with Exception Shielding)
> **Core Principle**:
> - When querying multiple microservices or student records concurrently, one failing call should not crash the entire batch.
> - Setting `return_exceptions=True` in `asyncio.gather()` captures errors as return values instead of abruptly stopping other tasks.

In [ ]:
# Practical Implementation: High-Throughput Async Service Aggregator
import asyncio
import time

async def query_student_microservice(student_id, service_name, delay, should_fail=False):
    await asyncio.sleep(delay)
    if should_fail:
        raise ConnectionError(f"Microservice '{service_name}' timed out!")
    return {"student_id": student_id, "source": service_name, "status": "ACTIVE", "latency": f"{delay*1000:.0f}ms"}

async def aggregate_student_profile(student_id):
    print(f"\nAggregating full profile for Student #{student_id} concurrently:")
    start = time.perf_counter()
    
    # Query Attendance, Fees, and Grades services concurrently
    responses = await asyncio.gather(
        query_student_microservice(student_id, "Attendance_Service", 0.05),
        query_student_microservice(student_id, "Fee_Ledger_Service", 0.08),
        query_student_microservice(student_id, "Legacy_Grading_DB", 0.03, should_fail=True),
        return_exceptions=True  # Shielding: Prevents one service crash from killing others
    )
    
    total_time = (time.perf_counter() - start) * 1000
    print(f"Aggregation finished in {total_time:.1f} ms")
    for r in responses:
        if isinstance(r, Exception):
            print(f"  [GRACEFUL FALLBACK] Caught Service Error: {r}")
        else:
            print(f"  [OK] Service: {r['source']:<22} -> Status: {r['status']}")

await aggregate_student_profile(101)

### 2. Async Task Timeout & Cancellation (`asyncio.wait_for`)
Enforcing strict SLAs: Automatically cancelling slow-running coroutines when latency budgets are exceeded.

In [ ]:
# SLA Timeout Enforcement Drill
async def long_running_query(query_name, duration):
    print(f"Executing '{query_name}' (expected: {duration}s)...")
    await asyncio.sleep(duration)
    return f"Query '{query_name}' completed successfully."

# Enforcing a 100ms timeout budget
try:
    result = await asyncio.wait_for(long_running_query("Fast Cache Lookup", 0.04), timeout=0.1)
    print("Result 1:", result)
except asyncio.TimeoutError:
    print("Query 1 timed out!")

try:
    result = await asyncio.wait_for(long_running_query("Slow Analytics Report", 0.5), timeout=0.1)
    print("Result 2:", result)
except asyncio.TimeoutError:
    print("Result 2: [TIMEOUT] Query exceeded SLA budget of 100ms and was cancelled cleanly!")